# 第三轮模型融合

比较同一份 UGV / USV 划分上的三种 YOLO26n。三组共用官方 COCO 初始化、数据、随机种子和训练设置。

| 组别 | 模型 | 相对上一组的改动 |
|---|---|---|
| A | YOLO26n 基线 | 两类检测头 |
| B | YOLO26n + P4 Mamba | 在第 13 层增加四方向扫描，分支宽度 32，状态维度 8 |
| C | YOLO26n + P4 Mamba + P5 ViT | 在 B 的第 10 层增加 ViT，嵌入维度 128，4 个注意力头 |

B 对 A、C 对 B 分别比较这两个分支在 seed 0 下的效果。与上一轮 s 版比较时，骨干规模和融合分支宽度都发生了变化。Pi 5 的 5 FPS 对应每张图 200 ms，需用训练后的权重在板上计时。

在项目根目录打开本文件，内核选 `.venv-cuda\Scripts\python.exe`。可先点 **全部运行**。正式训练时，到第 4 节复制所选模型的 PowerShell 命令。

## 1. 检查项目环境与数据划分

In [ ]:
from pathlib import Path
from tempfile import NamedTemporaryFile
import subprocess
import sys

PROJECT = Path.cwd().resolve()
if not (PROJECT / "train.py").is_file():
    raise RuntimeError("请从 yolo26-vit 项目根目录打开 Notebook。")
sys.path.insert(0, str(PROJECT / "training/support"))
from fusion.runtime import ARTIFACTS, STUDIES, bootstrap, read_json, sha256
bootstrap()

import torch
import yaml
from IPython.display import Markdown, display
from fusion.data import inspect_data
from fusion.models import build_model, load_model, make_detector
from ultralytics.nn.tasks import yaml_model_load

DATA = PROJECT / "training/configs/study-data.yaml"
data = inspect_data(DATA)
assert data["sha256"] == "760e212a7ba42aaf20b2f912aaa688c72c01bbc6c4eacd7af771866161605e8b", "数据划分与前两轮不同"
assert data["names"] == {0: "UGV", 1: "USV"}
OUTPUT = STUDIES / data["sha256"] / "phase3_n"
print("Python", sys.executable)
print("训练图", data["counts"]["train"]["images"], "  验证图", data["counts"]["val"]["images"], "  类别", data["names"])

RuntimeError: 请从 yolo26-vit 项目根目录打开 Notebook。

## 2. 准备官方 YOLO26n 权重

首次运行会下载 Ultralytics v8.4.0 的 `yolo26n.pt`。后续运行校验并复用本地文件。

In [ ]:
WEIGHTS = ARTIFACTS / "weights/yolo26n.pt"
subprocess.run([sys.executable, "train.py", "--download-weights", "yolo26n"], cwd=PROJECT, check=True)
print(WEIGHTS)
print("SHA-256", sha256(WEIGHTS))

D:\yolo26-vit\training\artifacts\weights\yolo26n.pt
SHA-256 9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef


## 3. 导出三组结构，核对权重迁移与前向输出

配置保存到 `models/fusion_phase3.yaml`，训练器读取同一个文件。先逐张量核对 COCO 权重，再把 80 类模型重建为两类。核对重建后只有分类头允许出现尺寸差异。表中参数量对应最终的两类模型，输出末维的 6 项为框坐标、置信度和类别编号。

In [ ]:
CONFIG = PROJECT / "models/fusion_phase3.yaml"
base = yaml_model_load("yolo26n.yaml")
p4 = dict(scan_layer=13, width=32, state_size=8, pretrained=False)
specs = {
    "baseline": dict(pretrained=False),
    "p4_mamba": p4,
    "p4_mamba_p5_vit": dict(p4, dim=128, heads=4),
}
configs = {name: dict(base, fusion_variant=name, fusion_spec=spec) for name, spec in specs.items()}
with NamedTemporaryFile("w", encoding="utf-8", dir=CONFIG.parent, delete=False) as file:
    yaml.safe_dump(configs, file, sort_keys=False)
Path(file.name).replace(CONFIG)
configs = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
print(CONFIG)

D:\yolo26-vit\models\fusion_phase3.yaml


In [ ]:
def check_structure(name, cfg, donor):
    wrapper, report = build_model(name, seed=0, yolo_weights=WEIGHTS, architecture=cfg)
    initialized = wrapper.model
    source, target = donor.state_dict(), initialized.state_dict()
    assert initialized.model[-1].nc == 80
    assert not report["yolo"]["unused_source"]
    assert all(torch.equal(target[k], source[v]) for k, v in report["yolo"]["loaded"].items())

    model = make_detector(initialized.yaml, nc=2, weights=initialized).eval()
    head = model.model[-1]
    transfer = model.transfer
    prefixes = (f"model.{head.i}.cv3.", f"model.{head.i}.one2one_cv3.")
    assert head.nc == 2 and transfer["unmatched"] and all(k.startswith(prefixes) for k in transfer["unmatched"])
    rebuilt = model.state_dict()
    assert all(torch.equal(rebuilt[k], target[v]) for k, v in transfer["loaded"].items())
    p4_layer, p5_layer = model.model[13], model.model[10]
    if "scan_layer" in cfg["fusion_spec"]:
        assert (p4_layer.in_proj.out_channels, p4_layer.state_size) == (32, 8)
    if "dim" in cfg["fusion_spec"]:
        assert (p5_layer.dim, p5_layer.encoder.self_attention.num_heads) == (128, 4)
    channels = [getattr(layer, "base", layer).cv2.conv.out_channels for layer in (p4_layer, p5_layer)]
    assert channels == [128, 256]
    with torch.inference_mode():
        boxes = model(torch.zeros(1, 3, 640, 640))[0]
    assert tuple(boxes.shape) == (1, 300, 6) and torch.isfinite(boxes).all()
    count = sum(p.numel() for p in model.parameters())
    return f"| {name} | {count:,} | {channels[0]} / {channels[1]} | {transfer['matched_tensors']} / {len(rebuilt)} | {tuple(boxes.shape)} |"

donor = load_model(WEIGHTS).model.float()
rows = [check_structure(name, cfg, donor) for name, cfg in configs.items()]
display(Markdown("\n".join(["| 模型 | 两类参数量 | P4 / P5 通道 | 两类重建时迁移的张量 | 输出形状 |",
                            "|---|---:|---|---|---|", *rows])))

Overriding model.yaml nc=80 with nc=2


Overriding model.yaml nc=80 with nc=2


Overriding model.yaml nc=80 with nc=2


| 模型 | 两类参数量 | P4 / P5 通道 | 两类重建时迁移的张量 | 输出形状 |
|---|---:|---|---|---|
| baseline | 2,504,580 | 128 / 256 | 606 / 708 | (1, 300, 6) |
| p4_mamba | 2,517,157 | 128 / 256 | 620 / 722 | (1, 300, 6) |
| p4_mamba_p5_vit | 2,781,350 | 128 / 256 | 637 / 739 | (1, 300, 6) |

## 4. 分别训练三个模型

每组都有一段完整命令。把想训练的那一段复制到 Windows PowerShell 运行，即可单独启动对应模型。三组均使用 GPU 0、640×640、batch 4、seed 0、AdamW、FP32，训练 100 轮。每组的结果保存到各自的目录，供第 5 节汇总。

### A · YOLO26n 基线

```powershell
Set-Location -LiteralPath 'D:\yolo26-vit'
$env:YOLO_DISABLE_CUDNN = '1'
$env:CUDA_LAUNCH_BLOCKING = '1'
& .\.venv-cuda\Scripts\python.exe train.py --model baseline `
    --model-set models/fusion_phase3.yaml --weights training/artifacts/weights/yolo26n.pt `
    --data training/configs/study-data.yaml --imgsz 640 --batch 4 --seed 0 --epochs 100 --patience 0 --device 0 `
    --run-dir training/artifacts/studies/760e212a7ba42aaf20b2f912aaa688c72c01bbc6c4eacd7af771866161605e8b/phase3_n/baseline
```

### B · YOLO26n + P4 Mamba

```powershell
Set-Location -LiteralPath 'D:\yolo26-vit'
$env:YOLO_DISABLE_CUDNN = '1'
$env:CUDA_LAUNCH_BLOCKING = '1'
& .\.venv-cuda\Scripts\python.exe train.py --model p4_mamba `
    --model-set models/fusion_phase3.yaml --weights training/artifacts/weights/yolo26n.pt `
    --data training/configs/study-data.yaml --imgsz 640 --batch 4 --seed 0 --epochs 100 --patience 0 --device 0 `
    --run-dir training/artifacts/studies/760e212a7ba42aaf20b2f912aaa688c72c01bbc6c4eacd7af771866161605e8b/phase3_n/p4_mamba
```

### C · YOLO26n + P4 Mamba + P5 ViT

```powershell
Set-Location -LiteralPath 'D:\yolo26-vit'
$env:YOLO_DISABLE_CUDNN = '1'
$env:CUDA_LAUNCH_BLOCKING = '1'
& .\.venv-cuda\Scripts\python.exe train.py --model p4_mamba_p5_vit `
    --model-set models/fusion_phase3.yaml --weights training/artifacts/weights/yolo26n.pt `
    --data training/configs/study-data.yaml --imgsz 640 --batch 4 --seed 0 --epochs 100 --patience 0 --device 0 `
    --run-dir training/artifacts/studies/760e212a7ba42aaf20b2f912aaa688c72c01bbc6c4eacd7af771866161605e8b/phase3_n/p4_mamba_p5_vit
```

## 5. 汇总最佳权重在验证集上的 mAP

训练完成后重跑本节。每组目录中的 `weights/best.pt`、`weights/last.pt`、`console.log` 和 `results.csv` 分别保存权重与日志。表格读取训练器重新验证 `best.pt` 生成的 `metrics.json`，mAP 范围为 0–1。

In [ ]:
summary = ["| 模型 | 状态 | 范围 | mAP50 | mAP50–95 |", "|---|---|---|---:|---:|"]
for name in configs:
    folder = OUTPUT / name
    run_file = folder / "run.json"
    status = read_json(run_file)["status"] if run_file.is_file() else "待训练"
    if status != "completed":
        summary.append(f"| {name} | {status} | — | — | — |")
        continue
    metrics = read_json(folder / "metrics.json")
    assert metrics["split"] == "val" and metrics["data_sha256"] == data["sha256"]
    assert metrics["checkpoint"] == "weights/best.pt" and metrics["checkpoint_sha256"] == sha256(folder / "weights/best.pt")
    for label, values in [("整体", metrics["overall"]), *metrics["per_class"].items()]:
        scores = ["—" if values[k] is None else f"{values[k]:.4f}" for k in ("map50", "map50_95")]
        summary.append(f"| {name} | 已完成 | {label} | {scores[0]} | {scores[1]} |")
display(Markdown("\n".join(summary)))
print("第三轮结果目录", OUTPUT)

| 模型 | 状态 | 范围 | mAP50 | mAP50–95 |
|---|---|---|---:|---:|
| baseline | 待训练 | — | — | — |
| p4_mamba | 待训练 | — | — | — |
| p4_mamba_p5_vit | 待训练 | — | — | — |

第三轮结果目录 D:\yolo26-vit\training\artifacts\studies\760e212a7ba42aaf20b2f912aaa688c72c01bbc6c4eacd7af771866161605e8b\phase3_n
